- Note down the accuracies for the following set of experiments on the given NN and compare the results Do the required modifications needed. Take training data percentage 30%, test data percentage 70%.
1. NN model with 2 hidden layers
    - Iris dataset
        - a. No. of epochs=100,
            -  check accuracy using activation functions Sigmoid, ReLu, Tanh
            -  check accuracy using optimizer sgd, ADAM
            -  check accuracy by varying learning rate in sgd as 0.0001, 0.0005, 5.
            - check accuracy using loss mean squared error, categorical cross entropy.
        - (b) No. of epochs =300
              - (i) Repeat the same above variations
    - (2) Ionosphere data
        - (a) Repeat the same settings as Iris

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn import datasets
from keras.models import Sequential
from keras.layers import Dense, Activation
from tensorflow.keras.optimizers import SGD, Adam
from sklearn.model_selection import train_test_split

c:\Users\hp\.conda\envs\example\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [17]:
def build_and_compile_model(input_dim, num_classes, activation_fn, optimizer_name, learning_rate, loss_fn ):
    """
    Dynamically  builds and compiles a 2 hidden layer neural network
    """
    if optimizer_name.lower() == 'sgd':
        optimizer = SGD(learning_rate = learning_rate)
    elif optimizer_name.lower()== 'adam':
        optimizer = Adam(learning_rate= learning_rate)
    else:
        raise ValueError(f"Unknown optimzer: {optimizer_name}")

    # Determine the number of output units
    # Binary classification expects 1 unit when using binary_crossentropy
    output_units = 1 if num_classes == 2 else num_classes

    # Build the sequential network
    model = Sequential([
        # Hidden layer 1
        Dense(16, activation = activation_fn, input_shape=(input_dim,)),
        #Hidden layer 2
        Dense(12, activation = activation_fn),
        #output layer
        #Dense(num_classes, activation = 'softmax')
        Dense(output_units, activation = 'sigmoid')
    ])

    # Model Compilation
    model.compile( optimizer= optimizer, loss = loss_fn, metrics=['accuracy'])

    return model


In [18]:
def run_experiment(X_train, X_test, y_train, y_test, epochs, activation, optimizer, lr, loss):
    input_dim= X_train.shape[1]

    # target count
    num_classes = len(np.unique(y_train))
    # configure model 
    model = build_and_compile_model(
        input_dim = input_dim,
        num_classes= num_classes,
        activation_fn = activation,
        optimizer_name = optimizer,
        learning_rate = lr,
        loss_fn = loss
    )

    # train the model
    model.fit(X_train, y_train, epochs = epochs, batch_size = 16, verbose = 0)

    # Evaluate performance
    loss_val, accuracy = model.evaluate(X_test, y_test, verbose = 0)

    #print(f"[{optimizer.upper()}] | {activation} | LR: {lr} | Loss: {loss} -> Test Accuracy: {accuracy:.4f}")

    return accuracy

In [3]:
# Preparing the data
# Iris Data

iris= datasets.load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.70, random_state = 42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [16]:
# Testing 100 epochs activations

print ("--------- EXPERIMENT: TESTING ACTIVATIONS ----------")
activations = ['sigmoid', 'relu', 'tanh']

for act in activations :
    run_experiment(X_train, X_test, y_train, y_test,
                    epochs = 100, activation = act, optimizer ='sgd', lr =0.01, loss = 'sparse_categorical_crossentropy')

--------- EXPERIMENT: TESTING ACTIVATIONS ----------


c:\Users\hp\.conda\envs\example\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[SGD] | sigmoid | LR: 0.01 | Loss: sparse_categorical_crossentropy -> Test Accuracy: 0.4190
[SGD] | relu | LR: 0.01 | Loss: sparse_categorical_crossentropy -> Test Accuracy: 0.9429
[SGD] | tanh | LR: 0.01 | Loss: sparse_categorical_crossentropy -> Test Accuracy: 0.8857


In [20]:
print("=========== EXPERIMENT: TESTING OPTIMIZERS ===============" )
optimizers = ['sgd', 'adam']

for opt in optimizers:
    run_experiment(X_train, X_test, y_train, y_test, 
                    epochs = 100, activation = 'relu', optimizer = opt, lr = 0.01, loss='sparse_categorical_crossentropy')

=========== EXPERIMENT: TESTING OPTIMIZERS ===============


c:\Users\hp\.conda\envs\example\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


[SGD] | relu | LR: 0.01 | Loss: sparse_categorical_crossentropy -> Test Accuracy: 0.8095
[ADAM] | relu | LR: 0.01 | Loss: sparse_categorical_crossentropy -> Test Accuracy: 0.9619


##### LOOPS ARE MANY - SWITCH TO GRID SEARCH 


In [6]:
# All variations
epochs_options = [100, 300]
activations = ['sigmoid', 'relu', 'tanh']
optimizers = ['sgd', 'adam']
learning_rates = [0.0001, 0.0005, 0.01, 0.1]
losses = ['sparse_categorical_crossentropy']

# Tracking best model
best_accuracy = 0.0
best_config = {}

print('------- STARTING MASTER GRID SEARCH FOR ALL COMBINATIONS ----------')
print(f"{'Epochs':<8} | {'Activation':<10} | {'Optimizer':<10} | {'LR':<8} | {'Loss Function':<32} | {'Test Accuracy':<8}")
print("-"*80)

for epoch in epochs_options:
    for act in activations:
        for opt in optimizers:
            for lr in learning_rates:
                for loss in losses:

                    # Run the specific combo
                    accuracy = run_experiment(
                        X_train, X_test, y_train, y_test,
                        epochs = epoch,
                        activation = act,
                        optimizer = opt,
                        lr = lr,
                        loss = loss
                    )

                    #Print results
                    print(f"{epoch:<8} | {act:<10} | {opt:<10} | {lr:<8} | {loss:<32} | {accuracy:.4f}")

                    # Best params
                    if accuracy > best_accuracy:
                        best_accuracy = accuracy
                        best_config = {
                            'epochs': epoch, 'activation': act,
                            'optimizer':opt, 'lr':lr , 'loss':loss
                        }

print("\n --------- EXPERIMENT COMPLETE -----------------")
print(f"Highest Accuracy Achieved: {best_accuracy:.4f}")
print(f"Best Configuration Found: {best_config}")

------- STARTING MASTER GRID SEARCH FOR ALL COMBINATIONS ----------
Epochs   | Activation | Optimizer  | LR       | Loss Function                    | Test Accuracy
--------------------------------------------------------------------------------


c:\Users\hp\.conda\envs\example\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


100      | sigmoid    | sgd        | 0.0001   | sparse_categorical_crossentropy  | 0.3143
100      | sigmoid    | sgd        | 0.0005   | sparse_categorical_crossentropy  | 0.3048
100      | sigmoid    | sgd        | 0.01     | sparse_categorical_crossentropy  | 0.4286
100      | sigmoid    | sgd        | 0.1      | sparse_categorical_crossentropy  | 0.9333
100      | sigmoid    | adam       | 0.0001   | sparse_categorical_crossentropy  | 0.3810
100      | sigmoid    | adam       | 0.0005   | sparse_categorical_crossentropy  | 0.4190
100      | sigmoid    | adam       | 0.01     | sparse_categorical_crossentropy  | 0.9619
100      | sigmoid    | adam       | 0.1      | sparse_categorical_crossentropy  | 0.9714
100      | relu       | sgd        | 0.0001   | sparse_categorical_crossentropy  | 0.4000
100      | relu       | sgd        | 0.0005   | sparse_categorical_crossentropy  | 0.3714
100      | relu       | sgd        | 0.01     | sparse_categorical_crossentropy  | 0.8286
100      |

##### Ionosphere data


In [13]:
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
# Fetch the Ionosphere dataset
ionosphere = fetch_openml(name="ionosphere", version=1, as_frame=True)

# Separate features (X) and targets (y)
X = ionosphere.data
y = ionosphere.target
le = LabelEncoder()
y = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.70, random_state = 42)

y_train = y_train.reshape(-1,1)
y_test = y_test.reshape(-1,1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
y

array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [19]:
# All variations
epochs_options = [100, 300]
activations = ['sigmoid', 'relu', 'tanh']
optimizers = ['sgd', 'adam']
learning_rates = [0.0001, 0.0005, 0.01, 0.1]
losses = ['binary_crossentropy']

# Tracking best model
best_accuracy = 0.0
best_config = {}

print('------- STARTING MASTER GRID SEARCH FOR ALL COMBINATIONS ----------')
print(f"{'Epochs':<8} | {'Activation':<10} | {'Optimizer':<10} | {'LR':<8} | {'Loss Function':<32} | {'Test Accuracy':<8}")
print("-"*80)

for epoch in epochs_options:
    for act in activations:
        for opt in optimizers:
            for lr in learning_rates:
                for loss in losses:

                    # Run the specific combo
                    accuracy = run_experiment(
                        X_train, X_test, y_train, y_test,
                        epochs = epoch,
                        activation = act,
                        optimizer = opt,
                        lr = lr,
                        loss = loss
                    )

                    #Print results
                    print(f"{epoch:<8} | {act:<10} | {opt:<10} | {lr:<8} | {loss:<32} | {accuracy:.4f}")

                    # Best params
                    if accuracy > best_accuracy:
                        best_accuracy = accuracy
                        best_config = {
                            'epochs': epoch, 'activation': act,
                            'optimizer':opt, 'lr':lr , 'loss':loss
                        }

print("\n --------- EXPERIMENT COMPLETE -----------------")
print(f"Highest Accuracy Achieved: {best_accuracy:.4f}")
print(f"Best Configuration Found: {best_config}")

------- STARTING MASTER GRID SEARCH FOR ALL COMBINATIONS ----------
Epochs   | Activation | Optimizer  | LR       | Loss Function                    | Test Accuracy
--------------------------------------------------------------------------------
100      | sigmoid    | sgd        | 0.0001   | binary_crossentropy              | 0.6341
100      | sigmoid    | sgd        | 0.0005   | binary_crossentropy              | 0.6341
100      | sigmoid    | sgd        | 0.01     | binary_crossentropy              | 0.6341
100      | sigmoid    | sgd        | 0.1      | binary_crossentropy              | 0.8293
100      | sigmoid    | adam       | 0.0001   | binary_crossentropy              | 0.6992
100      | sigmoid    | adam       | 0.0005   | binary_crossentropy              | 0.8008
100      | sigmoid    | adam       | 0.01     | binary_crossentropy              | 0.8415
100      | sigmoid    | adam       | 0.1      | binary_crossentropy              | 0.8577
100      | relu       | sgd       